# WavTRGAN: training on CelebA-HQ (corrected)

Trains WavTRGAN from scratch, keeps the best checkpoint on a **validation split**, and ends with the corrected test evaluation.

**What was fixed or added compared with the old training notebook**

| Problem in the old notebook | Fix |
|---|---|
| Gradient penalty used `torch.randn` (points far outside the real–fake line) | `torch.rand`, uniform in [0, 1] as in WGAN-GP |
| Gradient penalty interpolated `[real_b, real_b]` vs `[fake_b, fake_b]`, which the discriminator never sees | interpolates the real pairs `(input, ground truth)` and fake pairs `(input, output)` |
| Gradient penalty sent gradients into the generator (no `detach`) | fake pair detached in the D step |
| Loss weights, optimizer and losses differed from the paper (L1 ×1, perceptual ×0.1 MSE, no SSIM loss, lr 2e-4/1e-4, β2 0.999) | defaults follow the paper: L1 ×10, perceptual ×0.6 (L1 distance), SSIM ×0.4, style ×250, GAN ×0.1, lr 1e-4, β = (0.5, 0.9). Set `CFG["preset"] = "old_code"` to reproduce the old settings |
| No validation split (the paper says weights were chosen on one) | 1,000 training images held out; best checkpoint chosen by validation PSNR, never by test |
| `seed` was parsed but never used | seeds set for Python, NumPy and PyTorch |
| Test images loaded with random crop / flip / shuffle | validation and test use a fixed resize, no augmentation |
| One `print` per iteration (7,000 lines per epoch) | log every `print_every` iterations, per-epoch CSV log |
| No resume across Kaggle's 12 h limit | full checkpoint `last.pth` every epoch, automatic stop after `max_hours`, `resume` path |
| BatchNorm in the discriminator made the gradient penalty ~700 (×10), drowning the real/fake loss | no gradient penalty (`w_gp=0`); PatchGAN with spectral norm + InstanceNorm. Tried and rejected: spectral norm only (collapsed to D = 0.5 everywhere) and BatchNorm. With BatchNorm and with InstanceNorm the discriminator soon wins (D(real) ≈ 0.97, D(fake) ≈ 0.03); lower `lr_d` (e.g. 2e-5) to balance it. Learning rates follow CONFIG on resume |
| Xavier init with gain 0.02 on all 105 M weights: the output starts almost black and the first epochs are spent learning the brightness | PyTorch default initialisation (`init`) |
| Style-loss Gram matrix overflows in half precision | Gram computed in float32 (only matters if `amp=True`) |
| Only `use_wavelet` and `n_blocks` could be ablated | CONFIG switches `wavelet` (basis), `wave_levels`, `fusion` (directional IDWT of Eq. (3) or plain sub-band concatenation) and `d_spectral_norm` |
| Hole metrics used a threshold estimate of the corrupted region; no composited metrics | exact masks from `test/mask` when the dataset has them (`make_corrupted_dataset.py` writes them); PSNR/SSIM after compositing; per-image CSV and JSON summary per method |
| Resuming re-seeded every random generator, so each new Kaggle session replayed the data order of epoch 1 | random-generator states saved in `last.pth` and restored on resume (older checkpoints resume as before) |

**Runs for the revision.** Every run uses `preset = "paper"` and the dataset with exact masks from `wavtrgan-make-dataset.ipynb`. Give each run its own `out_dir` (e.g. `/kaggle/working/seed0`). Each ablation changes one setting of the seed-0 run, so `seed0` is their reference.

| Run | CONFIG change | Reviewer comment |
|---|---|---|
| `seed0`, `seed1`, `seed2` | `seed = 0`, `1`, `2` | 6: mean ± std of the main results |
| `no_l1`, `no_perc`, `no_ssim`, `no_style`, `no_gan` | one of `w_l1`, `w_perc`, `w_ssim`, `w_style`, `w_gan` set to 0 | 5: loss terms |
| `concat` | `fusion = "concat"` | 5: Eq. (3) vs. plain sub-band concatenation |
| `haar`, `db2` | `wavelet = "haar"` or `"db2"` | 5: wavelet basis |
| `levels2` | `wave_levels = 2` | 5: DWT levels |
| `no_sn` | `d_spectral_norm = False` | 5: discriminator regularization |
| `no_wavelet`, `depth3`, `depth6` | `use_wavelet = False`; `n_blocks = 3` or `6` | wavelet block and depth, if the paper reports them |

A difference from `seed0` smaller than about twice the std of the three seeds is within run-to-run noise.

**Kaggle:** GPU (T4 ×2 or P100), Internet on. Edit the CONFIG cell, then *Run all*.
To continue after a session ends: add the previous run's output as a dataset and set `CFG["resume"]` to its `last.pth`.

In [ ]:
!pip install -q pytorch_wavelets pytorch_msssim lpips torch-fidelity

## Configuration

In [ ]:
# =============================================================================================
# CONFIG
# =============================================================================================
CFG = dict(
    data_root   = "/kaggle/input/celeba-hq-img-full/CelebA-HQ-img",   # train/{a,b,mask}, test/{a,b,mask}; a = clean, b = corrupted
                                 # -> the dataset with exact masks from wavtrgan-make-dataset.ipynb
    direction   = "b2a",
    out_dir     = "/kaggle/working/run",
    resume      = "",            # path to a previous last.pth to continue training ("" = start from scratch)
    reset_d     = False,         # with resume: keep the generator, start a new discriminator (after changing d_norm / w_gp)
    seed        = 0,             # use 0, 1, 2 for the three runs (mean ± std in the paper)

    # model / ablations
    use_wavelet = True,          # False = "w/o wavelet" ablation
    n_blocks    = 12,            # transformer depth (3 / 6 / 12 for the depth ablation)
    wavelet     = "db3",         # wavelet basis of the wavelet block (db3 in the paper; e.g. "haar", "db2", "sym4")
    wave_levels = 1,             # DWT levels (1 in the paper; 2 for the level ablation)
    fusion      = "idwt",        # "idwt" = directional IDWT fusion of Eq. (3); "concat" = plain sub-band concatenation

    # schedule
    epochs       = 40,           # total epochs
    decay_epochs = 10,           # learning rate decays linearly to 0 over the last decay_epochs (0 = fixed lr)
    batch_size   = 4,
    lr_g = 1e-4, lr_d = 1e-4, beta1 = 0.5, beta2 = 0.9,
    preset = "paper",            # "paper" (weights below) or "old_code" (the settings of the old notebook)
    w_l1 = 10.0, w_perc = 0.6, w_ssim = 0.4, w_style = 250.0, w_gan = 0.1, w_gp = 0.0,
    init = "default",            # "default" (PyTorch init) or "xavier_0.02" (old notebook: starts from an almost black output)
    d_norm = "instance",         # discriminator normalisation with spectral norm: "instance", "batch" or "none"
    d_spectral_norm = True,      # False = discriminator without spectral norm (regularization ablation)
    amp = True,                  # mixed precision (T4 tensor cores): much faster; False = full float32

    # validation / checkpoints
    val_size    = 1000,          # images held out from train/ for model selection
    val_eval_n  = 300,           # validation images scored after every epoch
    save_every  = 5,             # also keep netG_epoch_N.pth every N epochs
    max_hours   = 11.0,          # stop cleanly (after saving) before Kaggle's 12 h limit
    print_every = 200,
    workers     = 4,
    multi_gpu   = True,          # DataParallel when 2 GPUs are available

    # final test evaluation
    final_eval  = True,
    final_lpips = True,
    final_fid   = True,
    pretrained_vgg = True,       # ImageNet VGG weights for the perceptual / style losses
)
if CFG["preset"] == "old_code":
    CFG.update(w_l1=1.0, w_perc=0.1, w_ssim=0.0, w_style=250.0, w_gan=0.1, w_gp=10.0,
               lr_g=2e-4, lr_d=1e-4, beta2=0.999, perc_dist="mse", d_norm="batch", init="xavier_0.02")
CFG.setdefault("perc_dist", "l1")
if CFG["preset"] != "paper":
    print("WARNING: preset %r. Every WavTRGAN number of the revision must come from one configuration, the "
          "\"paper\" preset described in the manuscript: do not mix these results with the revision runs." % CFG["preset"])

import os, random, time, json, csv
import numpy as np
import torch
os.makedirs(CFG["out_dir"], exist_ok=True)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"]); torch.cuda.manual_seed_all(CFG["seed"])
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(CFG, indent=1))
json.dump(CFG, open(os.path.join(CFG["out_dir"], "config.json"), "w"), indent=1)

## Data

In [ ]:
# =============================================================================================
# Data: train (augmented) / validation and test (fixed, no augmentation)
# =============================================================================================
from os.path import join
from PIL import Image
import torch.utils.data as data
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

EXTS = (".png", ".jpg", ".jpeg")
def image_names(folder):
    return sorted(f for f in os.listdir(folder) if f.lower().endswith(EXTS))

class PairDataset(data.Dataset):
    """Returns (corrupted, clean, name). train=True: resize 286 + random 256 crop + flip (as in the old notebook)."""
    def __init__(self, folder, names, train, direction="b2a"):
        self.a, self.b, self.names, self.train, self.direction = join(folder, "a"), join(folder, "b"), names, train, direction

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        n = self.names[i]
        a = Image.open(join(self.a, n)).convert("RGB")
        b = Image.open(join(self.b, n)).convert("RGB")
        if self.train:
            a, b = a.resize((286, 286), Image.BICUBIC), b.resize((286, 286), Image.BICUBIC)
            x, y = random.randint(0, 286 - 256), random.randint(0, 286 - 256)
            a, b = a.crop((x, y, x + 256, y + 256)), b.crop((x, y, x + 256, y + 256))
            if random.random() < 0.5:
                a, b = a.transpose(Image.FLIP_LEFT_RIGHT), b.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            a, b = a.resize((256, 256), Image.BICUBIC), b.resize((256, 256), Image.BICUBIC)
        a, b = transforms.functional.to_tensor(a), transforms.functional.to_tensor(b)    # [0, 1]
        return (b, a, n) if self.direction == "b2a" else (a, b, n)

train_dir, test_dir = join(CFG["data_root"], "train"), join(CFG["data_root"], "test")
if not os.path.isdir(join(test_dir, "mask")):
    print("WARNING: %s has no mask/ folder, so the test hole metrics will use a threshold estimate. "
          "Build the dataset with wavtrgan-make-dataset.ipynb to get exact masks." % test_dir)
all_train = image_names(join(train_dir, "a"))
rng = random.Random(1234)                                  # the split does not depend on the training seed
val_names = sorted(rng.sample(all_train, min(CFG["val_size"], len(all_train) // 5)))
val_set_names = set(val_names)
train_names = [n for n in all_train if n not in val_set_names]
open(os.path.join(CFG["out_dir"], "val_names.txt"), "w").write("\n".join(val_names))

train_set = PairDataset(train_dir, train_names, train=True, direction=CFG["direction"])
val_set = PairDataset(train_dir, val_names[:CFG["val_eval_n"]], train=False, direction=CFG["direction"])
loader_gen = torch.Generator(); loader_gen.manual_seed(CFG["seed"])
train_loader = DataLoader(train_set, batch_size=CFG["batch_size"], shuffle=True, drop_last=True, generator=loader_gen,
                          num_workers=CFG["workers"], pin_memory=torch.cuda.is_available(), persistent_workers=CFG["workers"] > 0)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=CFG["workers"])
print("train %d | validation %d (scored each epoch: %d) | test %d" % (
    len(train_names), len(val_names), len(val_set), len(image_names(join(test_dir, "a")))))

## Network (your WavTRGAN code; `USE_WAVELET` and `N_TRANSFORMER_BLOCKS` come from CONFIG)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # used inside the network code
USE_WAVELET = CFG["use_wavelet"]        # ablation switches (CONFIG cell)
N_TRANSFORMER_BLOCKS = CFG["n_blocks"]
WAVELET, WAVE_LEVELS, FUSION = CFG["wavelet"], CFG["wave_levels"], CFG["fusion"]

import math


from collections import OrderedDict
from pytorch_wavelets import DWTForward, DWTInverse
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F

import copy
#import logging
import math

from os.path import join as pjoin

import torch
import torch.nn as nn
import numpy as np

from torch.nn import CrossEntropyLoss, Dropout, Softmax, Linear, Conv2d, LayerNorm
from torch.nn.modules.utils import _pair
from scipy import ndimage

######################Wavelets_function############################
##################################################################



###########################################
#########Resnet Network###################
#########################################

class StdConv2d(nn.Conv2d):

    def forward(self, x):
        w = self.weight
        v, m = torch.var_mean(w, dim=[1, 2, 3], keepdim=True, unbiased=False)
        w = (w - m) / torch.sqrt(v + 1e-5)
        return F.conv2d(x, w, self.bias, self.stride, self.padding,
                        self.dilation, self.groups)


def conv3x3(cin, cout, stride=1, groups=1, bias=False):
    return StdConv2d(cin, cout, kernel_size=3, stride=stride,
                     padding=1, bias=bias, groups=groups)


def conv1x1(cin, cout, stride=1, bias=False):
    return StdConv2d(cin, cout, kernel_size=1, stride=stride,
                     padding=0, bias=bias)


class PreActBottleneck(nn.Module):
    """Pre-activation (v2) bottleneck block.
    """

    def __init__(self, cin, cout=None, cmid=None, stride=1):
        super().__init__()
        cout = cout or cin
        cmid = cmid or cout//4

        self.gn1 = nn.GroupNorm(32, cmid, eps=1e-6)
        self.conv1 = conv1x1(cin, cmid, bias=False)
        self.gn2 = nn.GroupNorm(32, cmid, eps=1e-6)
        self.conv2 = conv3x3(cmid, cmid, stride, bias=False)  # Original code has it on conv1!!
        self.gn3 = nn.GroupNorm(32, cout, eps=1e-6)
        self.conv3 = conv1x1(cmid, cout, bias=False)
        self.relu = nn.ReLU(inplace=True)

        if (stride != 1 or cin != cout):
            # Projection also with pre-activation according to paper.
            self.downsample = conv1x1(cin, cout, stride, bias=False)
            self.gn_proj = nn.GroupNorm(cout, cout)

    def forward(self, x):

        # Residual branch
        residual = x
        if hasattr(self, 'downsample'):
            residual = self.downsample(x)
            residual = self.gn_proj(residual)

        # Unit's branch
        y = self.relu(self.gn1(self.conv1(x)))
        y = self.relu(self.gn2(self.conv2(y)))
        y = self.gn3(self.conv3(y))

        y = self.relu(residual + y)
        return y

class ResNetV2(nn.Module):
    """Implementation of Pre-activation (v2) ResNet mode."""

    def __init__(self, block_units, width_factor):
        super().__init__()
        width = int(64 * width_factor)
        self.width = width

        self.root = nn.Sequential(OrderedDict([
            ('conv', StdConv2d(3, width, kernel_size=7, stride=2, bias=False, padding=3)),
            ('gn',nn.GroupNorm(32, width, eps=1e-6)),
            ('relu', nn.ReLU(inplace=True)),
            #('pool', nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        ]))

        self.body = nn.Sequential(OrderedDict([
            ('block1', nn.Sequential(OrderedDict(
                [('unit1', PreActBottleneck(cin=width, cout=width*4, cmid=width))] +
                [(f'unit{i:d}', PreActBottleneck(cin=width*4, cout=width*4, cmid=width)) for i in range(2, block_units[0] + 1)],
                ))),
            ('block2', nn.Sequential(OrderedDict(
                [('unit1', PreActBottleneck(cin=width*4, cout=width*8, cmid=width*2, stride=2))] +
                [(f'unit{i:d}', PreActBottleneck(cin=width*8, cout=width*8, cmid=width*2)) for i in range(2, block_units[1] + 1)],
                ))),
            ('block3', nn.Sequential(OrderedDict(
                [('unit1', PreActBottleneck(cin=width*8, cout=width*16, cmid=width*4, stride=2))] +
                [(f'unit{i:d}', PreActBottleneck(cin=width*16, cout=width*16, cmid=width*4)) for i in range(2, block_units[2] + 1)],
                ))),
            
        ]))

    def forward(self, x):
        features = []
        b, c, in_size, _ = x.size()
        x = self.root(x)
        features.append(x)
        x = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)(x)
        for i in range(len(self.body)-1):
            x = self.body[i](x)
            right_size = int(in_size / 4 / (i+1))
            if x.size()[2] != right_size:
                pad = right_size - x.size()[2]
                assert pad < 3 and pad > 0, "x {} should {}".format(x.size(), right_size)
                feat = torch.zeros((b, x.size()[1], right_size, right_size), device=x.device)
                feat[:, :, 0:x.size()[2], 0:x.size()[3]] = x[:]
            else:
                feat = x
            features.append(feat)
        x = self.body[-1](x)
        return x, features[::-1]
    
#####################################################
############Vision Transformer Net###################
####################################################
################ Wavlet_Suery#######################


class SSL(nn.Module):
    """Wavelet block: one DWT, one convolution per sub-band, then the directional fusion of Eq. (3).

    fusion "idwt" (paper): for each direction, the inverse DWT of the approximation and of that direction's
    detail sub-bands, with the other two directions set to zero.
    fusion "concat" (ablation): the same processed sub-bands, upsampled to the input size and concatenated
    (approximation + that direction's details) instead of the inverse DWT.
    wave / levels: wavelet basis and number of DWT levels. With levels > 1 the detail convolutions are
    shared across levels, so every setting has the same number of parameters.
    """
    def __init__(self, channels, wave="db3", levels=1, fusion="idwt"):
        super(SSL, self).__init__()
        if fusion not in ("idwt", "concat"):
            raise ValueError("fusion must be 'idwt' or 'concat', not %r" % fusion)
        self.wave, self.levels, self.fusion = wave, levels, fusion

        # Convolutional layers for processing wavelet components
        self.conv_approx = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.conv_horiz = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.conv_vert = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.conv_diag = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):   # pytorch_wavelets needs float32 (amp)
            return self._forward(x.float())

    def _forward(self, x):
        # models saved before these options existed have no such attributes: they used db3, 1 level, IDWT
        wave, levels = getattr(self, "wave", "db3"), getattr(self, "levels", 1)
        fusion = getattr(self, "fusion", "idwt")

        # Apply DWT to decompose the input image
        yl, yh = DWTForward(J=levels, mode='zero', wave=wave).to(x.device)(x)

        # Process each wavelet component with CNNs. details[d][j]: direction d at level j,
        # d = 0 horizontal, 1 vertical, 2 diagonal details
        approx_features = self.conv_approx(yl)
        convs = (self.conv_horiz, self.conv_vert, self.conv_diag)
        details = [[conv(h[:, :, d, :, :]) for h in yh] for d, conv in enumerate(convs)]

        if fusion == "concat":
            up = lambda t: F.interpolate(t, size=x.shape[-2:], mode="bilinear", align_corners=False)
            return tuple(torch.cat([up(approx_features)] + [up(t) for t in details[d]], dim=1) for d in range(3))

        # Reconstruct each direction with the inverse wavelet transform, the other two directions set to zero
        ifm = DWTInverse(wave=wave, mode='zero').to(x.device)
        return tuple(
            ifm((approx_features, [torch.stack([t if k == d else torch.zeros_like(t) for k in range(3)], dim=2)
                                   for t in details[d]]))
            for d in range(3))

#################################################

def swish(x):
    return x * torch.sigmoid(x)


ACT2FN = {"gelu": torch.nn.functional.gelu, "relu": torch.nn.functional.relu, "swish": swish}


class Attention(nn.Module):
    def __init__(self, vis):
        super(Attention, self).__init__()
        self.vis = vis
        self.num_attention_heads = 12
        self.attention_head_size = int(768 / self.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        #self.query = SSL(768)
        self.query = Linear(768, self.all_head_size)
        self.key = Linear(768, self.all_head_size)
        self.value = Linear(768, self.all_head_size)

        self.out = Linear(768, 768)
        self.attn_dropout = Dropout(0.0)
        self.proj_dropout = Dropout(0.1)

        self.softmax = Softmax(dim=-1)

 
    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)
    
    def forward(self, hidden_states):
       
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        
        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_probs = self.softmax(attention_scores)
        weights = attention_probs if self.vis else None
        attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        attention_output = self.proj_dropout(attention_output)
        return attention_output, weights

class Mlp(nn.Module):
    def __init__(self):
        super(Mlp, self).__init__()
        self.fc1 = Linear(768, 3072)
        self.fc2 = Linear(3072, 768)
        self.act_fn = ACT2FN["gelu"]
        self.dropout = Dropout(0.1)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.normal_(self.fc1.bias, std=1e-6)
        nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act_fn(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class Embeddings(nn.Module):
    """Construct the embeddings from patch, position embeddings.
    """
    def __init__(self, img_size, in_channels=3):
        super(Embeddings, self).__init__()
        self.num_layers = (3, 4, 9)
        self.width_factor = 1
        img_size = _pair(img_size)

        grid_size = (16,16)
        patch_size = (img_size[0] // 16 // grid_size[0], img_size[1] // 16 // grid_size[1])
        patch_size_real = (patch_size[0] * 16, patch_size[1] * 16)
        n_patches = (img_size[0] // patch_size_real[0]) * (img_size[1] // patch_size_real[1])  
        
        self.hybrid_model = ResNetV2(block_units=self.num_layers, width_factor=self.width_factor)
        in_channels = self.hybrid_model.width * 16
        
        self.patch_embeddings = Conv2d(in_channels=in_channels,
                                       out_channels=768,
                                       kernel_size=patch_size,
                                       stride=patch_size)
        self.position_embeddings = nn.Parameter(torch.zeros(1, n_patches, 768))

        self.dropout = Dropout(0.1)

        if USE_WAVELET:
            self.wavelet = SSL(3, WAVELET, WAVE_LEVELS, FUSION)
        # input + one direction image (6 channels); with fusion "concat": input + approximation + one detail per level
        in_ch = 6 if FUSION == "idwt" else 3 * (2 + WAVE_LEVELS)
        self.conv_layer = nn.Conv2d(in_channels=in_ch, out_channels=3, kernel_size=1, stride=1, padding=0).to(device)
        self.conv_con = nn.Conv2d(in_channels=1024, out_channels=3, kernel_size=1, stride=1, padding=0).to(device)
    def forward(self, x):
        if not hasattr(self, 'wavelet'):      # ablation: no wavelet block, one ResNet pass on the input
            x, features = self.hybrid_model(x)
            x = self.patch_embeddings(x)
            embeddings = x.flatten(2).transpose(-1, -2) + self.position_embeddings
            return self.dropout(embeddings), features

        x_h, x_v, x_sp = self.wavelet(x)

        #extract horizontal features by ResNet
        x_h = torch.cat((x, x_h),1)
        x_h = self.conv_layer(x_h)
        x_h, features = self.hybrid_model(x_h)
    
        #extract vertical features by ResNet
        x_v = torch.cat((x, x_v),1)
        x_v = self.conv_layer(x_v)
        x_v, features = self.hybrid_model(x_v)

        #extract diagonal features by ResNet
        x_sp = torch.cat((x, x_sp),1)
        x_sp = self.conv_layer(x_sp)
        x_sp, features = self.hybrid_model(x_sp)

        x = x_h +x_v +x_sp
        
        x = self.patch_embeddings(x)  # (B, hidden. n_patches^(1/2), n_patches^(1/2))
        x = x.flatten(2)
        x = x.transpose(-1, -2)  # (B, n_patches, hidden)

        embeddings = x + self.position_embeddings
        embeddings = self.dropout(embeddings)
        return embeddings, features


class Block(nn.Module):
    def __init__(self, vis):
        super(Block, self).__init__()
        self.hidden_size = 768
        self.attention_norm = LayerNorm(768, eps=1e-6)
        self.ffn_norm = LayerNorm(768, eps=1e-6)
        self.ffn = Mlp()
        self.attn = Attention(vis)

    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x = x + h

        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x = x + h
        return x, weights

class Encoder(nn.Module):
    def __init__(self, vis):
        super(Encoder, self).__init__()
        self.vis = vis
        self.layer = nn.ModuleList()
        self.encoder_norm = LayerNorm(768, eps=1e-6)
        for _ in range(N_TRANSFORMER_BLOCKS):  #number of block of tranformers layer
            layer = Block(vis)
            self.layer.append(copy.deepcopy(layer))

    def forward(self, hidden_states):
        attn_weights = []
        for layer_block in self.layer:
            hidden_states, weights = layer_block(hidden_states)
            if self.vis:
                attn_weights.append(weights)
        encoded = self.encoder_norm(hidden_states)
        return encoded, attn_weights


class Transformer(nn.Module):
    def __init__(self, img_size, vis):
        super(Transformer, self).__init__()
        self.embeddings = Embeddings(img_size=img_size)
        self.encoder = Encoder(vis)

    def forward(self, input_ids):
        embedding_output, features = self.embeddings(input_ids)
        encoded, attn_weights = self.encoder(embedding_output)  # (B, n_patch, hidden)
        return encoded, attn_weights, features


class Conv2dReLU(nn.Sequential):
    def __init__(
            self,
            in_channels,
            out_channels,
            kernel_size,
            padding=0,
            stride=1,
            use_batchnorm=True,
    ):
        conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            bias=not (use_batchnorm),
        )
        relu = nn.ReLU(inplace=True)

        bn = nn.BatchNorm2d(out_channels)

        super(Conv2dReLU, self).__init__(conv, bn, relu)


class DecoderBlock(nn.Module):
    def __init__(
            self,
            in_channels,
            out_channels,
            skip_channels=3,
            use_batchnorm=True,
    ):
        super().__init__()
        self.conv1 = Conv2dReLU(
            in_channels + skip_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            use_batchnorm=use_batchnorm,
        )
        self.conv2 = Conv2dReLU(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            use_batchnorm=use_batchnorm,
        )
        self.up = nn.UpsamplingBilinear2d(scale_factor=2)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class SegmentationHead(nn.Sequential):

    def __init__(self, in_channels, out_channels, kernel_size=3, upsampling=1):
        conv2d = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2)
        upsampling = nn.UpsamplingBilinear2d(scale_factor=upsampling) if upsampling > 1 else nn.Identity()
        super().__init__(conv2d, upsampling)


class DecoderCup(nn.Module):
    def __init__(self):
        super().__init__()
        head_channels = 512
        self.conv_more = Conv2dReLU(
            768,
            head_channels,
            kernel_size=3,
            padding=1,
            use_batchnorm=True,
        )

        self.projection_layer = nn.Conv2d(768, 512, kernel_size=1)
        decoder_channels = (256, 128, 64, 16)
        in_channels = [head_channels] + list(decoder_channels[:-1])
        out_channels = decoder_channels
        skip_channels = [512, 256, 64, 16]
        self.n_skip = 3
        for i in range(4-self.n_skip):  # re-select the skip channels according to n_skip
            skip_channels[3-i]=0

        blocks = [
            DecoderBlock(in_ch, out_ch, sk_ch) for in_ch, out_ch, sk_ch in zip(in_channels, out_channels, skip_channels)
        ]
        self.blocks = nn.ModuleList(blocks)

    def forward(self, hidden_states, features=None):
        B, n_patch, hidden = hidden_states.size()  # reshape from (B, n_patch, hidden) to (B, h, w, hidden)
        h, w = int(np.sqrt(n_patch)), int(np.sqrt(n_patch))
        x = hidden_states.permute(0, 2, 1)
        x = x.contiguous().view(B, hidden, h, w)
        
        x = self.conv_more(x)
  
        for i, decoder_block in enumerate(self.blocks):
            if features is not None:
                skip = features[i] if (i < self.n_skip) else None
            else:
                skip = None
            x = decoder_block(x, skip=skip)
        return x

#////////////////////////////////////////////
#Using Transformers as Encoder
#///////////////////////////////////////////
class VisionTransformer(nn.Module):
    def __init__(self, img_size=256, num_classes=21843, zero_head=False, vis=False):
        super(VisionTransformer, self).__init__()
        self.zero_head = zero_head
        self.transformer = Transformer(img_size, vis)
        self.decoder = DecoderCup()
        self.segmentation_head = SegmentationHead(
            in_channels=16,
            out_channels=3,
            kernel_size=3,
        )
        
    def forward(self, x):
        if x.size()[1] == 1:
            x = x.repeat(1,3,1,1)
       
        #x_input = x
        #x = self.wavelet_transform(x)
        #x = x + x_input
        
        x, attn_weights, features = self.transformer(x)  # (B, n_patch, hidden)
        x = self.decoder(x, features)
        
        x = self.segmentation_head(x)
        return x

## Discriminator and GAN utilities

In [ ]:
# =============================================================================================
# Discriminator (PatchGAN + spectral norm), GAN loss, initialisation, lr schedule
# =============================================================================================
import functools
import torch.nn as nn
from torch.nn import init

def get_norm_layer(norm_type="batch"):
    if norm_type == "none":
        return None
    if norm_type == "batch":
        return functools.partial(nn.BatchNorm2d, affine=True)
    if norm_type == "instance":
        return functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)
    raise NotImplementedError(norm_type)

def init_weights(net, init_type="xavier", gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, "weight") and (classname.find("Conv") != -1 or classname.find("Linear") != -1):
            if init_type == "normal":
                init.normal_(m.weight.data, 0.0, gain)
            elif init_type == "xavier":
                init.xavier_normal_(m.weight.data, gain=gain)
            elif init_type == "kaiming":
                init.kaiming_normal_(m.weight.data, a=0, mode="fan_in")
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find("BatchNorm2d") != -1:
            init.normal_(m.weight.data, 1.0, gain)
            init.constant_(m.bias.data, 0.0)
    net.apply(init_func)
    return net

class NLayerDiscriminator(nn.Module):
    def __init__(self, input_nc, ndf=64, n_layers=3, norm_layer=nn.BatchNorm2d, spectral_norm=True):
        super().__init__()
        if norm_layer is None:                  # no normalisation layer
            use_bias, norm_layer = True, lambda c: nn.Identity()
        elif type(norm_layer) == functools.partial:
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d
        sn = nn.utils.spectral_norm if spectral_norm else (lambda m: m)
        seq = [sn(nn.Conv2d(input_nc, ndf, 4, 2, 1)), nn.LeakyReLU(0.2, True)]
        nf_mult = 1
        for n in range(1, n_layers):
            nf_prev, nf_mult = nf_mult, min(2 ** n, 8)
            seq += [sn(nn.Conv2d(ndf * nf_prev, ndf * nf_mult, 4, 2, 1, bias=use_bias)), norm_layer(ndf * nf_mult), nn.LeakyReLU(0.2, True)]
        nf_prev, nf_mult = nf_mult, min(2 ** n_layers, 8)
        seq += [sn(nn.Conv2d(ndf * nf_prev, ndf * nf_mult, 4, 1, 1, bias=use_bias)), norm_layer(ndf * nf_mult), nn.LeakyReLU(0.2, True)]
        seq += [sn(nn.Conv2d(ndf * nf_mult, 1, 4, 1, 1))]
        self.model = nn.Sequential(*seq)

    def forward(self, x):
        return self.model(x)

class GANLoss(nn.Module):
    """LSGAN: MSE to 1 (real) / 0 (fake)."""
    def __init__(self):
        super().__init__()
        self.loss = nn.MSELoss()

    def __call__(self, pred, target_is_real):
        return self.loss(pred, torch.full_like(pred, 1.0 if target_is_real else 0.0))

def gradient_penalty(D, real_ab, fake_ab):
    """WGAN-GP on conditional pairs; alpha ~ U[0, 1]; computed in float32."""
    alpha = torch.rand(real_ab.size(0), 1, 1, 1, device=real_ab.device)
    x = (alpha * real_ab.float() + (1 - alpha) * fake_ab.float()).requires_grad_(True)
    with torch.autocast(device_type=x.device.type, enabled=False):
        d = D(x)
    grad = torch.autograd.grad(d.sum(), x, create_graph=True)[0]
    return ((grad.flatten(1).norm(2, dim=1) - 1) ** 2).mean()

def linear_decay(epochs, decay_epochs):
    """lr factor for epoch e (0-based): 1 until the last decay_epochs, then linear to 0."""
    keep = epochs - decay_epochs
    return lambda e: 1.0 if e < keep or decay_epochs == 0 else max(0.0, 1.0 - (e - keep + 1) / float(decay_epochs + 1))

## Losses

In [ ]:
# =============================================================================================
# Losses: perceptual (VGG-16), style (VGG-19 Gram, float32), SSIM
# =============================================================================================
import torchvision
import torchvision.models as models
from pytorch_msssim import ssim as ssim_fn

def _vgg(name):
    weights = {"vgg16": "IMAGENET1K_V1", "vgg19": "IMAGENET1K_V1"}[name] if CFG["pretrained_vgg"] else None
    return getattr(models, name)(weights=weights).features.eval()

class VGGPerceptualLoss(nn.Module):
    """VGG-16 relu1_2, relu2_2, relu3_3, relu4_3 on ImageNet-normalised, 224-resized images."""
    def __init__(self, dist="l1"):
        super().__init__()
        f = _vgg("vgg16")
        self.blocks = nn.ModuleList([f[:4], f[4:9], f[9:16], f[16:23]])
        for p in self.parameters():
            p.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.dist = nn.L1Loss() if dist == "l1" else nn.MSELoss()

    def forward(self, x, y):
        x = F.interpolate((x - self.mean) / self.std, size=(224, 224), mode="bilinear", align_corners=False)
        y = F.interpolate((y - self.mean) / self.std, size=(224, 224), mode="bilinear", align_corners=False)
        loss = 0.0
        for block in self.blocks:
            x, y = block(x), block(y)
            loss = loss + self.dist(x, y)
        return loss / len(self.blocks)

class StyleLoss(nn.Module):
    """Gram-matrix L1 on VGG-19 relu2_2, relu3_4, relu4_4, relu5_2 (same layers as the old notebook)."""
    def __init__(self):
        super().__init__()
        f = _vgg("vgg19")
        self.slices = nn.ModuleList([f[:9], f[9:18], f[18:27], f[27:32]])
        for p in self.parameters():
            p.requires_grad = False
        self.criterion = nn.L1Loss()

    @staticmethod
    def compute_gram(x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            b, ch, h, w = x.size()
            f = x.view(b, ch, h * w)
            return f.bmm(f.transpose(1, 2)) / (h * w * ch)

    def forward(self, x, y):
        loss = 0.0
        for s in self.slices:
            x, y = s(x), s(y)
            loss = loss + self.criterion(self.compute_gram(x), self.compute_gram(y))
        return loss

def ssim_loss(pred, target):
    return 1 - ssim_fn(pred.float().clamp(0, 1), target.float(), data_range=1.0, size_average=True)

## Corrected evaluation functions (same as the evaluation notebook)

In [ ]:
# =============================================================================================
# Corrected evaluation for the WavTRGAN revision (one notebook cell, self-contained)
#
# Part 1  generate_wavtrgan(): runs WavTRGAN on the test pairs WITHOUT random crop / flip and
#         saves input, ground truth and output under the SAME file name (needed for fair,
#         reproducible pairs and for hole-only metrics). If the test set has exact masks
#         (test_root/mask, written by make_corrupted_dataset.py), they are saved too: give them
#         to the non-blind baselines and to evaluate(mask_dir=...).
# Part 2  evaluate(): scores every method on the same pairs:
#         - PSNR / SSIM on float images (no uint8 overflow), full image, RGB
#         - the same after compositing (output inside the hole, input outside)
#         - PSNR / SSIM on the corrupted region only (hole) and PSNR on the rest (known)
#         - LPIPS (AlexNet), FID (torch-fidelity)
#         The hole is the exact mask when mask_dir is given. Without it, the hole is estimated as the
#         pixels where input and ground truth differ by > HOLE_THRESHOLD, which misses corrupted
#         pixels whose fill looks like the original: use exact masks for the paper.
#         Files are matched by NAME (sorted), never by glob order.
#         Besides out_csv (one row per method), it writes per method <out_csv>_<method>_per_image.csv
#         and <out_csv>_<method>.json: the JSON files are the input of evaluation/aggregate.py
#         (mean +/- std over seeds), the per-image CSVs that of evaluation/pick_examples.py.
#
# Folder layout expected for every method: one folder of outputs whose file names are the same
# as in GT_DIR / INPUT_DIR (e.g. 0001.png). Run baselines on INPUT_DIR and save their outputs
# under the same names.
# =============================================================================================
import os, csv, json
import numpy as np
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

EXTS = (".png", ".jpg", ".jpeg")
HOLE_THRESHOLD = 0.05          # without exact masks: a pixel is "corrupted" if input and ground truth differ by > 0.05


def load_rgb(path, size=None):
    im = Image.open(path).convert("RGB")          # also drops the alpha channel of plt.imsave PNGs
    if size is not None and im.size != size:
        im = im.resize(size, Image.BICUBIC)
    return np.asarray(im, dtype=np.float64)       # float64 in [0, 255]: no uint8 wrap-around


def load_mask(path, size):
    """Binary mask (True = corrupted) from a PNG with white holes, resized with nearest neighbour."""
    im = Image.open(path).convert("L")
    if im.size != size:
        im = im.resize(size, Image.NEAREST)
    return np.asarray(im) > 127


def names_in(folder):
    return sorted(f for f in os.listdir(folder) if f.lower().endswith(EXTS))


# ---------------------------------------------------------------------------------------------
# Part 1: WavTRGAN outputs on fixed pairs (run in the WavTRGAN notebook, after the model cells)
# ---------------------------------------------------------------------------------------------
@torch.no_grad()
def generate_wavtrgan(model, test_root, out_root, direction="b2a", size=256, device="cuda"):
    """test_root/{a,b}: clean / corrupted pairs with identical names (and optionally test_root/mask).
    Writes out_root/{input,gt,wavtrgan} and, when test_root/mask exists, out_root/mask."""
    corrupted_dir, clean_dir = (os.path.join(test_root, "b"), os.path.join(test_root, "a")) if direction == "b2a" \
        else (os.path.join(test_root, "a"), os.path.join(test_root, "b"))
    mask_src = os.path.join(test_root, "mask")
    keys = ("input", "gt", "wavtrgan") + (("mask",) if os.path.isdir(mask_src) else ())
    dirs = {k: os.path.join(out_root, k) for k in keys}
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    model = model.to(device).eval()
    for name in names_in(corrupted_dir):
        stem = os.path.splitext(name)[0] + ".png"
        inp = load_rgb(os.path.join(corrupted_dir, name), (size, size)) / 255.0
        gt = load_rgb(os.path.join(clean_dir, name), (size, size)) / 255.0
        x = torch.from_numpy(inp).permute(2, 0, 1)[None].float().to(device)   # [0, 1], as in training
        y = model(x).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
        for key, arr in (("input", inp), ("gt", gt), ("wavtrgan", y)):
            Image.fromarray((arr * 255).round().astype(np.uint8)).save(os.path.join(dirs[key], stem))
        if "mask" in dirs:
            mask = load_mask(os.path.join(mask_src, stem), (size, size))
            Image.fromarray(mask.astype(np.uint8) * 255).save(os.path.join(dirs["mask"], stem))
    print("saved", len(names_in(dirs["input"])), "pairs to", out_root)
    if "mask" not in dirs:
        print("WARNING: %s has no mask/ folder, so hole metrics will use the threshold estimate. "
              "Regenerate the test set with make_corrupted_dataset.py to get exact masks." % test_root)


# ---------------------------------------------------------------------------------------------
# Part 2: metrics for all methods on the same pairs
# ---------------------------------------------------------------------------------------------
def masked_psnr(gt, pr, m):
    """PSNR over the pixels where m is True (m: H x W bool), images float in [0, 255]."""
    if m.sum() == 0:
        return np.nan
    mse = ((gt - pr) ** 2)[m].mean()
    return np.inf if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)


def masked_ssim(smap, m):
    """Mean of an SSIM map (H x W x C, from structural_similarity(..., full=True)) over the pixels where m is True."""
    if m.sum() == 0:
        return np.nan
    return smap.mean(axis=2)[m].mean()


def ratio_bin(ratio):
    """10% bin of the hole ratio, labelled as in evaluation/evaluate.py (e.g. "30-40%")."""
    low = min(int(ratio * 10), 9) * 10
    return "%d-%d%%" % (low, low + 10)


def finite_mean(values):
    """Mean over finite values: a perfect pixel region (inf dB) or an empty hole (nan) is left out."""
    v = np.array(values, dtype=float)
    v = v[np.isfinite(v)]
    return float(v.mean()) if v.size else float("nan")


def evaluate(methods, gt_dir, input_dir, out_csv="metrics.csv", lpips_on=True, fid_on=True, device=None, mask_dir=None):
    """methods: {"WavTRGAN": folder, "CAML": folder, ...}. Every folder must contain the same file names.
    mask_dir: exact masks (white = corrupted) with the same names; None = threshold estimate."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    gt_names = names_in(gt_dir)
    mask_kind = "exact" if mask_dir else "threshold %.2f" % HOLE_THRESHOLD
    if not mask_dir:
        print("WARNING: no mask_dir, the hole is estimated from |input - gt| > %.2f. "
              "Use exact masks for the paper." % HOLE_THRESHOLD)
    lp = None
    if lpips_on:
        import lpips
        lp = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
    base = os.path.splitext(out_csv)[0]
    rows = []
    for method, folder in methods.items():
        stems = {os.path.splitext(f)[0]: f for f in names_in(folder)}
        pairs = [(g, stems[os.path.splitext(g)[0]]) for g in gt_names if os.path.splitext(g)[0] in stems]
        missing = len(gt_names) - len(pairs)
        assert pairs, "%s: no file names in common with %s" % (method, gt_dir)
        per_image, resized = [], 0
        for g, f in pairs:
            gt = load_rgb(os.path.join(gt_dir, g))
            size = (gt.shape[1], gt.shape[0])
            pr = load_rgb(os.path.join(folder, f))
            if pr.shape != gt.shape:              # e.g. a baseline that outputs another resolution
                pr = load_rgb(os.path.join(folder, f), size=size)
                resized += 1
            inp = load_rgb(os.path.join(input_dir, g), size=size)
            if mask_dir:
                hole = load_mask(os.path.join(mask_dir, os.path.splitext(g)[0] + ".png"), size)
            else:
                hole = np.abs(inp - gt).max(axis=2) > HOLE_THRESHOLD * 255
            comp = np.where(hole[..., None], pr, inp)   # output inside the hole, uncorrupted input outside
            ssim_val, smap = structural_similarity(gt, pr, data_range=255, channel_axis=2, full=True)   # one SSIM pass
            r = {"image": os.path.splitext(g)[0], "bin": ratio_bin(hole.mean()), "mask_ratio": float(hole.mean()),
                 "psnr_full": peak_signal_noise_ratio(gt, pr, data_range=255), "ssim_full": ssim_val,
                 "psnr_full_comp": peak_signal_noise_ratio(gt, comp, data_range=255),
                 "ssim_full_comp": structural_similarity(gt, comp, data_range=255, channel_axis=2),
                 "psnr_hole": masked_psnr(gt, pr, hole), "ssim_hole": masked_ssim(smap, hole),
                 "psnr_known": masked_psnr(gt, pr, ~hole)}
            if lp is not None:
                t = lambda a: torch.from_numpy(a / 127.5 - 1).permute(2, 0, 1)[None].float().to(device)
                with torch.no_grad():
                    r["lpips_full"] = lp(t(pr), t(gt)).item()
            per_image.append(r)

        keys = [k for k in per_image[0] if k not in ("image", "bin")]
        overall = {k: finite_mean([r[k] for r in per_image]) for k in keys}
        if fid_on:
            import torch_fidelity
            overall["fid_full"] = torch_fidelity.calculate_metrics(
                input1=folder, input2=gt_dir, cuda=torch.cuda.is_available(), fid=True, verbose=False
            )["frechet_inception_distance"]
        bins = {}
        for label in sorted({r["bin"] for r in per_image}):
            in_bin = [r for r in per_image if r["bin"] == label]
            bins[label] = {"n": len(in_bin), **{k: finite_mean([r[k] for r in in_bin]) for k in keys}}

        tag = base + "_" + "".join(c if c.isalnum() else "_" for c in method)
        with open(tag + "_per_image.csv", "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=list(per_image[0].keys()))
            w.writeheader()
            w.writerows(per_image)
        with open(tag + ".json", "w") as fh:
            json.dump({"method": method, "n_images": len(pairs),
                       "protocol": {"mask": mask_kind, "resized_outputs": resized,
                                    "ssim": "RGB, 7x7 uniform window (scikit-image default)"},
                       "overall": overall, "bins": bins}, fh, indent=2)

        row = {"method": method, "images": len(pairs), "missing": missing, "resized": resized, "mask": mask_kind,
               "PSNR": overall["psnr_full"], "SSIM": overall["ssim_full"],
               "PSNR_comp": overall["psnr_full_comp"], "SSIM_comp": overall["ssim_full_comp"],
               "PSNR_hole": overall["psnr_hole"], "SSIM_hole": overall["ssim_hole"],
               "PSNR_known": overall["psnr_known"], "LPIPS": overall.get("lpips_full", np.nan),
               "hole_ratio": overall["mask_ratio"]}
        if fid_on:
            row["FID"] = overall["fid_full"]
        rows.append(row)
        print("%-22s n=%d%s%s  PSNR %.2f  SSIM %.4f | composited: PSNR %.2f SSIM %.4f | hole: PSNR %.2f SSIM %.4f"
              " | known PSNR %.2f | LPIPS %.4f%s" % (
                  method, row["images"], " (%d missing!)" % missing if missing else "",
                  " (%d resized)" % resized if resized else "", row["PSNR"], row["SSIM"],
                  row["PSNR_comp"], row["SSIM_comp"], row["PSNR_hole"], row["SSIM_hole"], row["PSNR_known"],
                  row["LPIPS"], "  FID %.2f" % row["FID"] if "FID" in row else ""))
    with open(out_csv, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print("mean corrupted fraction of the test set: %.1f%% (%s mask)  -> %s" % (
        100 * rows[0]["hole_ratio"], mask_kind, out_csv))
    return rows


# ---------------------------------------------------------------------------------------------
# Sanity check: the old uint8 code vs this code on two unrelated images (should be ~8 dB here)
# ---------------------------------------------------------------------------------------------
def sanity_check():
    rng = np.random.default_rng(0)
    a = rng.integers(0, 256, (64, 64, 3)).astype(np.uint8)
    b = rng.integers(0, 256, (64, 64, 3)).astype(np.uint8)
    old = 20 * np.log10(255 / np.sqrt(np.mean((a - b) ** 2)))           # the old, wrong formula
    new = peak_signal_noise_ratio(a.astype(np.float64), b.astype(np.float64), data_range=255)
    print("unrelated random images: old uint8 PSNR %.2f dB (wrong), corrected %.2f dB" % (old, new))


## Training

In [ ]:
# =============================================================================================
# Training (resumable)
# =============================================================================================
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import torch.optim as optim

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

net_g = VisionTransformer(img_size=256, num_classes=9)
net_d = NLayerDiscriminator(6, 64, 3, get_norm_layer(CFG["d_norm"]), spectral_norm=CFG["d_spectral_norm"])
if CFG["init"] == "xavier_0.02":
    net_g, net_d = init_weights(net_g, "xavier", 0.02), init_weights(net_d, "xavier", 0.02)
net_g, net_d = net_g.to(device), net_d.to(device)
print("generator %.2f M parameters | discriminator %.2f M" % (
    sum(p.numel() for p in net_g.parameters()) / 1e6, sum(p.numel() for p in net_d.parameters()) / 1e6))

opt_g = optim.Adam(net_g.parameters(), lr=CFG["lr_g"], betas=(CFG["beta1"], CFG["beta2"]))
opt_d = optim.Adam(net_d.parameters(), lr=CFG["lr_d"], betas=(CFG["beta1"], CFG["beta2"]))
sched_g = optim.lr_scheduler.LambdaLR(opt_g, linear_decay(CFG["epochs"], CFG["decay_epochs"]))
sched_d = optim.lr_scheduler.LambdaLR(opt_d, linear_decay(CFG["epochs"], CFG["decay_epochs"]))
scaler_g = torch.cuda.amp.GradScaler(enabled=CFG["amp"])
scaler_d = torch.cuda.amp.GradScaler(enabled=CFG["amp"])

start_epoch, best, history = 1, {"psnr": -1.0, "epoch": 0}, []
resume_rng = None
if CFG["resume"]:
    ck = torch.load(CFG["resume"], map_location=device, weights_only=False)
    net_g.load_state_dict(ck["net_g"]); opt_g.load_state_dict(ck["opt_g"])
    sched_g.load_state_dict(ck["sched_g"]); sched_d.load_state_dict(ck["sched_d"])
    if CFG["amp"] == ck["cfg"].get("amp"):
        scaler_g.load_state_dict(ck["scaler_g"])
    if CFG["reset_d"]:
        print("new discriminator (reset_d=True); generator weights and optimizer restored")
    else:
        net_d.load_state_dict(ck["net_d"]); opt_d.load_state_dict(ck["opt_d"])
        if CFG["amp"] == ck["cfg"].get("amp"):
            scaler_d.load_state_dict(ck["scaler_d"])
    start_epoch, best, history = ck["epoch"] + 1, ck["best"], ck["history"]
    resume_rng = ck.get("rng")   # restored just before training, once every network is built
    prev_best = os.path.join(os.path.dirname(CFG["resume"]), "netG_best.pth")
    if os.path.exists(prev_best) and not os.path.exists(os.path.join(CFG["out_dir"], "netG_best.pth")):
        import shutil
        shutil.copy(prev_best, os.path.join(CFG["out_dir"], "netG_best.pth"))   # keep the best model of earlier sessions
    # learning rates always follow CONFIG, so lr_g / lr_d can be changed when resuming
    for opt_, sched_, base in ((opt_g, sched_g, CFG["lr_g"]), (opt_d, sched_d, CFG["lr_d"])):
        sched_.base_lrs = [base] * len(opt_.param_groups)
        for pg in opt_.param_groups:
            pg["initial_lr"] = base
            pg["lr"] = base * sched_.lr_lambdas[0](sched_.last_epoch)
    print("resumed from %s: continuing at epoch %d (best so far: %.2f dB at epoch %d) | lr G %.1e, D %.1e" % (
        CFG["resume"], start_epoch, best["psnr"], best["epoch"], opt_g.param_groups[0]["lr"], opt_d.param_groups[0]["lr"]))

if CFG["multi_gpu"] and torch.cuda.device_count() > 1:
    net_g, net_d = nn.DataParallel(net_g), nn.DataParallel(net_d)
    print("DataParallel on", torch.cuda.device_count(), "GPUs")

crit_gan = GANLoss().to(device)
crit_l1 = nn.L1Loss()
crit_perc = VGGPerceptualLoss(CFG["perc_dist"]).to(device)
crit_style = StyleLoss().to(device)

@torch.no_grad()
def validate(model):
    """PSNR / SSIM (RGB, float) and hole PSNR on the validation images; same definitions as the test metrics."""
    model.eval()
    ps, ss, ph = [], [], []
    for inp, gt, _ in val_loader:
        out = model(inp.to(device)).float().clamp(0, 1)
        o = (out[0].permute(1, 2, 0).cpu().numpy() * 255).round()
        t = (gt[0].permute(1, 2, 0).numpy() * 255).round()
        x = (inp[0].permute(1, 2, 0).numpy() * 255).round()
        ps.append(peak_signal_noise_ratio(t, o, data_range=255))
        ss.append(structural_similarity(t, o, data_range=255, channel_axis=2))
        hole = np.abs(x - t).max(axis=2) > 0.05 * 255
        if hole.any():
            ph.append(10 * np.log10(255.0 ** 2 / max(((t - o) ** 2)[hole].mean(), 1e-10)))
    model.train()
    return float(np.mean(ps)), float(np.mean(ss)), float(np.mean(ph)) if ph else float("nan")

def save_preview(model, epoch):
    inp, gt, _ = val_set[0]
    with torch.no_grad():
        model.eval()
        out = model(inp[None].to(device)).float().clamp(0, 1)[0].cpu()
        model.train()
    row = torch.cat([inp, out, gt], dim=2).permute(1, 2, 0).numpy()
    os.makedirs(os.path.join(CFG["out_dir"], "samples"), exist_ok=True)
    Image.fromarray((row * 255).round().astype(np.uint8)).save(os.path.join(CFG["out_dir"], "samples", "epoch_%03d.png" % epoch))

def save_checkpoint(epoch, path):
    torch.save({"epoch": epoch, "net_g": unwrap(net_g).state_dict(), "net_d": unwrap(net_d).state_dict(),
                "opt_g": opt_g.state_dict(), "opt_d": opt_d.state_dict(),
                "sched_g": sched_g.state_dict(), "sched_d": sched_d.state_dict(),
                "scaler_g": scaler_g.state_dict(), "scaler_d": scaler_d.state_dict(),
                "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                        "loader": loader_gen.get_state()},
                "best": best, "history": history, "cfg": CFG}, path + ".tmp")
    os.replace(path + ".tmp", path)                           # never leaves a half-written last.pth

def set_requires_grad(net, flag):
    for p in net.parameters():
        p.requires_grad = flag

if resume_rng:   # continue the random streams, so a resumed run does not replay the data order of epoch 1
    random.setstate(resume_rng["python"])
    np.random.set_state(resume_rng["numpy"])
    torch.set_rng_state(resume_rng["torch"].cpu())
    loader_gen.set_state(resume_rng["loader"].cpu())
    if torch.cuda.is_available() and len(resume_rng["cuda"]) == torch.cuda.device_count():
        torch.cuda.set_rng_state_all([s.cpu() for s in resume_rng["cuda"]])
t_start = time.time()
stopped_early = False
net_g.train(); net_d.train()
for epoch in range(start_epoch, CFG["epochs"] + 1):
    t_epoch = time.time()
    sums, n_it = {}, 0
    for it, (real_a, real_b, _) in enumerate(train_loader, 1):
        real_a, real_b = real_a.to(device, non_blocking=True), real_b.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=CFG["amp"]):
            fake_b = net_g(real_a)

        # (1) discriminator: LSGAN + gradient penalty on (input, image) pairs
        set_requires_grad(net_d, True)
        opt_d.zero_grad(set_to_none=True)
        real_ab = torch.cat((real_a, real_b), 1)
        fake_ab = torch.cat((real_a, fake_b.detach()), 1)
        with torch.autocast(device_type=device.type, enabled=CFG["amp"]):
            d_fake, d_real = net_d(fake_ab).float(), net_d(real_ab).float()
            loss_d_gan = crit_gan(d_fake, False) + crit_gan(d_real, True)
        gp = gradient_penalty(net_d, real_ab, fake_ab) if CFG["w_gp"] > 0 else torch.zeros((), device=device)
        loss_d = loss_d_gan + CFG["w_gp"] * gp
        scaler_d.scale(loss_d).backward()
        scaler_d.step(opt_d)
        scaler_d.update()

        # (2) generator
        set_requires_grad(net_d, False)
        opt_g.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=CFG["amp"]):
            l_gan = crit_gan(net_d(torch.cat((real_a, fake_b), 1)).float(), True)
            fb = fake_b.float()
            l_l1 = crit_l1(fb, real_b)
            l_perc = crit_perc(fb, real_b)
            l_style = crit_style(fb, real_b)
        l_ssim = ssim_loss(fb, real_b) if CFG["w_ssim"] > 0 else torch.zeros((), device=device)
        loss_g = (CFG["w_gan"] * l_gan + CFG["w_l1"] * l_l1 + CFG["w_perc"] * l_perc
                  + CFG["w_ssim"] * l_ssim + CFG["w_style"] * l_style)
        if not torch.isfinite(loss_g):
            print("non-finite generator loss at epoch %d iteration %d: skipped" % (epoch, it))
            continue
        scaler_g.scale(loss_g).backward()
        scaler_g.step(opt_g)
        scaler_g.update()

        n_it += 1
        for k, v in (("D", loss_d), ("Dreal", d_real.mean()), ("Dfake", d_fake.mean()), ("GP", gp), ("G", loss_g), ("gan", l_gan), ("l1", l_l1),
                     ("perc", l_perc), ("ssim", l_ssim), ("style", l_style)):
            sums[k] = sums.get(k, 0.0) + float(v.detach())
        if it % CFG["print_every"] == 0 or it == 1:
            print("Epoch[%d](%d/%d) " % (epoch, it, len(train_loader)) +
                  " ".join("%s %.4g" % (k, v / n_it) for k, v in sums.items()) +
                  " | %.2f s/it" % ((time.time() - t_epoch) / it))
        if (time.time() - t_start) / 3600 > CFG["max_hours"]:
            stopped_early = True
            break

    if stopped_early:
        print("max_hours reached in epoch %d: this epoch is not counted. Resume from last.pth (epoch %d)." % (epoch, epoch - 1))
        break

    sched_g.step(); sched_d.step()
    v_psnr, v_ssim, v_hole = validate(net_g)
    minutes = (time.time() - t_epoch) / 60
    row = {"epoch": epoch, "minutes": round(minutes, 2), "lr": opt_g.param_groups[0]["lr"],
           **{k: v / max(n_it, 1) for k, v in sums.items()},
           "val_psnr": v_psnr, "val_ssim": v_ssim, "val_psnr_hole": v_hole}
    history.append(row)
    save_preview(net_g, epoch)
    if v_psnr > best["psnr"]:
        best = {"psnr": v_psnr, "ssim": v_ssim, "psnr_hole": v_hole, "epoch": epoch}
        torch.save(unwrap(net_g), os.path.join(CFG["out_dir"], "netG_best.pth"))   # full model, as before
    if epoch % CFG["save_every"] == 0:
        torch.save(unwrap(net_g).state_dict(), os.path.join(CFG["out_dir"], "netG_epoch_%d.pth" % epoch))
    save_checkpoint(epoch, os.path.join(CFG["out_dir"], "last.pth"))
    with open(os.path.join(CFG["out_dir"], "log.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(history[-1].keys()))
        w.writeheader(); w.writerows(history)
    print("===> Epoch %d: %.1f min | lr %.2e | val PSNR %.2f SSIM %.4f PSNR-hole %.2f | best %.2f (epoch %d)" % (
        epoch, minutes, row["lr"], v_psnr, v_ssim, v_hole, best["psnr"], best["epoch"]))

print("training time this session: %.2f h" % ((time.time() - t_start) / 3600))

## Curves

In [ ]:
# Training curves
import matplotlib.pyplot as plt
if history:
    ep = [r["epoch"] for r in history]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(ep, [r["val_psnr"] for r in history], label="PSNR"); ax[0].plot(ep, [r["val_psnr_hole"] for r in history], label="PSNR hole")
    ax[0].set_title("validation PSNR (dB)"); ax[0].legend()
    ax[1].plot(ep, [r["val_ssim"] for r in history]); ax[1].set_title("validation SSIM")
    for k in ("l1", "perc", "style", "gan", "D"):
        ax[2].plot(ep, [r.get(k, np.nan) for r in history], label=k)
    ax[2].set_yscale("log"); ax[2].set_title("training losses"); ax[2].legend()
    for a in ax: a.set_xlabel("epoch")
    plt.tight_layout(); plt.show()
    print("total training time: %.1f h over %d epochs" % (sum(r["minutes"] for r in history) / 60, len(history)))

## Test evaluation

In [ ]:
# =============================================================================================
# Final test evaluation with the best validation checkpoint
# =============================================================================================
if CFG["final_eval"] and not stopped_early:
    best_path = os.path.join(CFG["out_dir"], "netG_best.pth")
    best_model = torch.load(best_path, map_location=device, weights_only=False) if os.path.exists(best_path) else unwrap(net_g)
    print("best validation epoch:", best["epoch"])
    eval_root = os.path.join(CFG["out_dir"], "eval")
    generate_wavtrgan(best_model, test_dir, eval_root, direction=CFG["direction"], size=256, device=device)
    mask_dir = os.path.join(eval_root, "mask")                     # exact masks, if the test set has them
    results = evaluate({"Input (no inpainting)": os.path.join(eval_root, "input"),
                        "WavTRGAN": os.path.join(eval_root, "wavtrgan")},
                       gt_dir=os.path.join(eval_root, "gt"), input_dir=os.path.join(eval_root, "input"),
                       mask_dir=mask_dir if os.path.isdir(mask_dir) else None,
                       out_csv=os.path.join(CFG["out_dir"], "test_metrics.csv"),
                       lpips_on=CFG["final_lpips"], fid_on=CFG["final_fid"], device=device)
else:
    print("final evaluation skipped (training not finished in this session)")